# 9. Parallel Execution (Fan-out/Fan-in)

The last core mechanic before tools and agents: running multiple nodes **simultaneously** on the same state, rather than choosing one path (Video 7) or repeating one path (Video 8). This is the clearest possible payoff of Video 6's reducers — multiple nodes writing to the same field *at the same time* is exactly the collision scenario `operator.add` exists to handle.

This also brings Priya's Applicant Screener to its **final form** (parallel checks, a verdict, three-way routing, Video 8's rescore loop and the summary note), closing out its arc from Video 3 and all of Module 1–2.

**Sections**
1. Recap: the utility + the Screener pipeline so far
2. Fan-out: multiple edges from one node, or a conditional edge returning a list
3. Fan-in: automatic synchronization, and ordering (non-)guarantees
4. The Applicant Screener, final form: three parallel criteria checks → a verdict node → Video 7's routing and Video 8's rescore loop → summary
5. Module 1–2 recap

## 1. Recap

Same `llm_utils.py` as every notebook since Video 3, and the same three candidates: Hermione, Harry and Ron. If you're running this notebook fresh, the cell below recreates the utility, the resumes and the parsing step.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

for env_var in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"]:
    if not os.environ.get(env_var):
        os.environ[env_var] = getpass.getpass(f"{env_var}: ")

In [ ]:
%%writefile llm_utils.py
"""Provider-agnostic LLM utility used by every node in this course from Video 3 onward.

Call `call_llm(provider, messages, system=None)` instead of ever importing a provider
SDK directly from graph/node code. Swapping models is then a one-line `provider` change.
"""
from dataclasses import dataclass

import anthropic
from openai import OpenAI
from google import genai
from google.genai import types as genai_types

# Standardize on one current, non-legacy model per provider.
CLAUDE_MODEL = "claude-sonnet-5"
GPT_MODEL = "gpt-5.1"
GEMINI_MODEL = "gemini-3.5-flash"

DEFAULT_PROVIDER = "claude"  # the rest of this course leaves this default in place

_anthropic_client = anthropic.Anthropic()
_openai_client = OpenAI()
_gemini_client = genai.Client()


@dataclass
class LLMResponse:
    content: str
    provider: str
    model: str
    raw: object  # the untouched provider response, for debugging


def _call_claude(messages: list[dict], system: str | None) -> LLMResponse:
    resp = _anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=system or anthropic.NOT_GIVEN,
        messages=messages,
    )
    # Newer Claude models may put a thinking block first, so join only the text blocks.
    text = "".join(block.text for block in resp.content if block.type == "text")
    return LLMResponse(content=text, provider="claude", model=CLAUDE_MODEL, raw=resp)


def _call_gpt(messages: list[dict], system: str | None) -> LLMResponse:
    full_messages = ([{"role": "system", "content": system}] if system else []) + messages
    resp = _openai_client.chat.completions.create(model=GPT_MODEL, messages=full_messages)
    return LLMResponse(content=resp.choices[0].message.content, provider="gpt", model=GPT_MODEL, raw=resp)


def _to_gemini_contents(messages: list[dict]) -> list[genai_types.Content]:
    role_map = {"user": "user", "assistant": "model"}
    return [
        genai_types.Content(role=role_map[m["role"]], parts=[genai_types.Part(text=m["content"])])
        for m in messages
    ]


def _call_gemini(messages: list[dict], system: str | None) -> LLMResponse:
    config = genai_types.GenerateContentConfig(system_instruction=system) if system else None
    resp = _gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=_to_gemini_contents(messages),
        config=config,
    )
    return LLMResponse(content=resp.text, provider="gemini", model=GEMINI_MODEL, raw=resp)


_PROVIDERS = {"claude": _call_claude, "gpt": _call_gpt, "gemini": _call_gemini}


def call_llm(provider: str = DEFAULT_PROVIDER, messages: list[dict] | None = None, system: str | None = None) -> LLMResponse:
    if provider not in _PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}, expected one of {list(_PROVIDERS)}")
    return _PROVIDERS[provider](messages or [], system)


In [ ]:
import operator
from typing import Annotated, TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

from llm_utils import call_llm

# The three candidates we follow through every notebook in this story.
RESUMES = {
    "Hermione": """
Hermione Granger
7 years of experience as a senior backend software engineer.
Skills: Python, distributed systems, SQL, PostgreSQL, Kubernetes, API design.
Led a 6-person platform team at a fintech company, mentoring junior engineers.
""",
    "Harry": """
Harry Potter
4 years of experience as a software developer.
Skills: Python, SQL, REST APIs.
Informally mentored two interns. No distributed systems or Kubernetes experience.
""",
    "Ron": """
Ron Weasley
8 months of experience as a junior web developer.
Skills: HTML, CSS, basic JavaScript.
No team or leadership experience.
""",
}

PARSE_SYSTEM_PROMPT = (
    "You extract structured fields from a resume. Reply with exactly four lines, "
    "no extra commentary:\n"
    "Name: <candidate name>\n"
    "Years: <years of experience as a number>\n"
    "Skills: <comma-separated skills>\n"
    "Leadership: <one short phrase on leadership or mentoring experience, or none>"
)


def show_graph(graph):
    """Render a compiled graph. Falls back to Mermaid text if the PNG renderer is unavailable."""
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_mermaid())


def parse_resume(state: dict) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": state["resume_text"]}], system=PARSE_SYSTEM_PROMPT)
    fields = {"candidate_name": "", "years_experience": "", "skills": "", "leadership": ""}
    for line in result.content.strip().splitlines():
        if line.startswith("Name:"):
            fields["candidate_name"] = line.removeprefix("Name:").strip()
        elif line.startswith("Years:"):
            fields["years_experience"] = line.removeprefix("Years:").strip()
        elif line.startswith("Skills:"):
            fields["skills"] = line.removeprefix("Skills:").strip()
        elif line.startswith("Leadership:"):
            fields["leadership"] = line.removeprefix("Leadership:").strip()
    fields["evaluation_notes"] = [f"Parsed fields for {fields['candidate_name'] or 'candidate'}."]
    return fields

## 2. Fan-out: multiple edges from one node, or a conditional edge returning a list

Two ways to express "run several nodes at once from here":
1. **Multiple plain `add_edge` calls out of the same node** — the most common case, used below for the Screener.
2. **A conditional edge whose router returns a *list* of node names** instead of a single string — useful when which branches run (not just how many) depends on runtime state.

A minimal demo of each, before applying this to the Screener:

In [ ]:
class FanState(TypedDict):
    results: Annotated[list[str], operator.add]


def branch_a(state: FanState) -> dict:
    return {"results": ["A"]}


def branch_b(state: FanState) -> dict:
    return {"results": ["B"]}


def branch_c(state: FanState) -> dict:
    return {"results": ["C"]}


def collect(state: FanState) -> dict:
    return {}


# Fan-out style 1: multiple add_edge calls from the same node.
fanout_builder = StateGraph(FanState)
fanout_builder.add_node("branch_a", branch_a)
fanout_builder.add_node("branch_b", branch_b)
fanout_builder.add_node("branch_c", branch_c)
fanout_builder.add_node("collect", collect)
fanout_builder.add_edge(START, "branch_a")
fanout_builder.add_edge(START, "branch_b")
fanout_builder.add_edge(START, "branch_c")
fanout_builder.add_edge("branch_a", "collect")
fanout_builder.add_edge("branch_b", "collect")
fanout_builder.add_edge("branch_c", "collect")
fanout_builder.add_edge("collect", END)

fanout_graph = fanout_builder.compile()
show_graph(fanout_graph)
sorted(fanout_graph.invoke({"results": []})["results"])

In [ ]:
# Fan-out style 2: a conditional edge whose router returns a LIST of node names.
# Here it's state-dependent - only branches matching a condition get included.
def pick_branches(state: FanState) -> list[str]:
    return ["branch_a", "branch_c"]  # branch_b skipped this time, decided at runtime


dynamic_builder = StateGraph(FanState)
dynamic_builder.add_node("branch_a", branch_a)
dynamic_builder.add_node("branch_b", branch_b)
dynamic_builder.add_node("branch_c", branch_c)
dynamic_builder.add_node("collect", collect)
dynamic_builder.add_conditional_edges(START, pick_branches, ["branch_a", "branch_b", "branch_c"])
dynamic_builder.add_edge("branch_a", "collect")
dynamic_builder.add_edge("branch_b", "collect")
dynamic_builder.add_edge("branch_c", "collect")
dynamic_builder.add_edge("collect", END)

dynamic_graph = dynamic_builder.compile()
sorted(dynamic_graph.invoke({"results": []})["results"])  # only A and C ran

## 3. Fan-in: automatic synchronization, and ordering (non-)guarantees

Notice `collect` above was never told how many branches to wait for — LangGraph figures that out from the graph structure and automatically waits for **every** incoming edge for a given step to finish before running the downstream node. That's fan-in: no API to call, just a consequence of how convergent edges are wired.

One caveat: branches that fan out together run concurrently, and **there's no guarantee about the order** in which their individual writes land, which is exactly why `results` above needed sorting to compare reliably and exactly why a plain-overwrite field would be a race condition waiting to happen. A reducer makes the *merge* order-independent even when individual branch completion order isn't.

## 4. The Applicant Screener, final form

**Priya's problem:** one rubric call gives one number, and she wants to see skills, experience and leadership judged separately, without waiting for three calls in a row.

- **Input:** a resume (we run all three candidates).
- **Output:** three sub-scores, a total `score`, a `decision`, a `summary` note for the hiring manager, and an `evaluation_notes` audit trail with an entry per check.

**The logic:** instead of one `score_against_rubric` node, fan out from `parse_resume` into **three independent, simultaneous** criteria-check nodes: skills match (0–40), experience match (0–40) and leadership (0–20). Each writes its own sub-score field and appends its own note to the shared `evaluation_notes` list (Video 6's reducer, finally earning its keep for real). A `verdict` node fans back in and adds the three sub-scores into an overall score. That score then goes through everything from the earlier videos: Video 7's `reject` / `interview` / `strong_match` routing, Video 8's rescore loop for borderline totals (50–64), and Video 4's summary note.

```
parse_resume -> [skills | experience | leadership] -> verdict -> (rescore loop) -> reject / interview / strong_match -> generate_summary
```

In [ ]:
SKILLS_PROMPT = (
    "Score 0-40: how well do this candidate's skills overlap with "
    "[Python, distributed systems, Kubernetes, SQL]? "
    "Reply with exactly: Skills score: <integer 0-40>"
)
EXPERIENCE_PROMPT = (
    "Score 0-40 for years of experience: 5 or more years earns up to 40 points, "
    "and fewer years earn proportionally fewer. "
    "Reply with exactly: Experience score: <integer 0-40>"
)
LEADERSHIP_PROMPT = (
    "Score 0-20: how strong is the leadership, mentoring or team-lead experience listed? "
    "Reply with exactly: Leadership score: <integer 0-20>"
)

MAX_RESCORE_ATTEMPTS = 2

RELAXED_RUBRIC = (
    "Re-score the candidate from 0-100 against this RELAXED rubric - give the benefit of the "
    "doubt on borderline cases:\n"
    "- 3+ years experience: up to 40 points\n"
    "- Any overlap with [Python, distributed systems, Kubernetes, SQL, backend, API]: up to 40 points\n"
    "- Any hint of leadership, mentorship, or team responsibility: up to 20 points\n"
    "A second look can lift a borderline candidate to the interview stage but never to a strong "
    "match, so the final score must not exceed 79. Do not mention this cap in your rationale.\n"
    "Reply with exactly two lines, no extra commentary:\n"
    "Score: <integer 0-79>\n"
    "Rationale: <one sentence>"
)


class ScreenerState(TypedDict):
    resume_text: str
    candidate_name: str
    years_experience: str
    skills: str
    leadership: str
    skills_score: int
    experience_score: int
    leadership_score: int
    score: int
    score_rationale: str
    decision: str
    summary: str
    rescore_attempts: Annotated[int, operator.add]
    evaluation_notes: Annotated[list[str], operator.add]


def _extract_score(text: str, prefix: str) -> int:
    for line in text.strip().splitlines():
        if line.startswith(prefix):
            return int("".join(ch for ch in line if ch.isdigit()) or 0)
    return 0


def candidate_summary(state: ScreenerState) -> str:
    return (
        f"Name: {state['candidate_name']}\n"
        f"Years of experience: {state['years_experience']}\n"
        f"Skills: {state['skills']}\n"
        f"Leadership: {state['leadership']}"
    )


def check_skills_match(state: ScreenerState) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary(state)}], system=SKILLS_PROMPT)
    score = _extract_score(result.content, "Skills score:")
    return {"skills_score": score, "evaluation_notes": [f"Skills check: {score}/40"]}


def check_experience_match(state: ScreenerState) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary(state)}], system=EXPERIENCE_PROMPT)
    score = _extract_score(result.content, "Experience score:")
    return {"experience_score": score, "evaluation_notes": [f"Experience check: {score}/40"]}


def check_leadership(state: ScreenerState) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary(state)}], system=LEADERSHIP_PROMPT)
    score = _extract_score(result.content, "Leadership score:")
    return {"leadership_score": score, "evaluation_notes": [f"Leadership check: {score}/20"]}


def verdict(state: ScreenerState) -> dict:
    total = state["skills_score"] + state["experience_score"] + state["leadership_score"]
    rationale = (
        f"skills={state['skills_score']}/40, experience={state['experience_score']}/40, "
        f"leadership={state['leadership_score']}/20"
    )
    return {"score": total, "score_rationale": rationale, "evaluation_notes": [f"Verdict: {total}/100 ({rationale})"]}


def adjust_and_rescore(state: ScreenerState) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary(state)}], system=RELAXED_RUBRIC)
    score, rationale = state["score"], state["score_rationale"]
    for line in result.content.strip().splitlines():
        if line.startswith("Score:"):
            score = int("".join(ch for ch in line if ch.isdigit()) or 0)
        elif line.startswith("Rationale:"):
            rationale = line.removeprefix("Rationale:").strip()
    attempt_number = state["rescore_attempts"] + 1
    return {
        "score": score,
        "score_rationale": rationale,
        "rescore_attempts": 1,  # delta - operator.add accumulates this across passes
        "evaluation_notes": [f"Rescore attempt {attempt_number}: {score}/100 ({rationale})"],
    }


def route_after_scoring(state: ScreenerState) -> str:
    if 50 <= state["score"] < 65 and state["rescore_attempts"] < MAX_RESCORE_ATTEMPTS:
        return "rescore"
    elif state["score"] < 50:
        return "reject"
    elif state["score"] < 80:
        return "interview"
    else:
        return "strong_match"


def reject_node(state: ScreenerState) -> dict:
    return {"decision": f"Not a match ({state['score']}/100): {state['score_rationale']}"}


def interview_node(state: ScreenerState) -> dict:
    return {"decision": f"Move to interview ({state['score']}/100): {state['score_rationale']}"}


def strong_match_node(state: ScreenerState) -> dict:
    return {"decision": f"Strong match ({state['score']}/100): {state['score_rationale']}"}


def generate_summary(state: ScreenerState) -> dict:
    prompt = (
        f"Candidate: {state['candidate_name']}\n"
        f"Decision: {state['decision']}\n\n"
        "Write a two-sentence plain-text note for the hiring manager explaining this decision and the suggested next step."
    )
    result = call_llm("claude", messages=[{"role": "user", "content": prompt}])
    return {"summary": result.content}


final_builder = StateGraph(ScreenerState)
final_builder.add_node("parse_resume", parse_resume)
final_builder.add_node("check_skills_match", check_skills_match)
final_builder.add_node("check_experience_match", check_experience_match)
final_builder.add_node("check_leadership", check_leadership)
final_builder.add_node("verdict", verdict)
final_builder.add_node("adjust_and_rescore", adjust_and_rescore)
final_builder.add_node("reject", reject_node)
final_builder.add_node("interview", interview_node)
final_builder.add_node("strong_match", strong_match_node)
final_builder.add_node("generate_summary", generate_summary)

final_builder.add_edge(START, "parse_resume")

# Fan-out: three simultaneous criteria checks off the same node.
final_builder.add_edge("parse_resume", "check_skills_match")
final_builder.add_edge("parse_resume", "check_experience_match")
final_builder.add_edge("parse_resume", "check_leadership")

# Fan-in: verdict waits for all three automatically.
final_builder.add_edge("check_skills_match", "verdict")
final_builder.add_edge("check_experience_match", "verdict")
final_builder.add_edge("check_leadership", "verdict")

# Video 7's routing and Video 8's rescore loop, now fed by the combined verdict.
branch_map = {"rescore": "adjust_and_rescore", "reject": "reject", "interview": "interview", "strong_match": "strong_match"}
final_builder.add_conditional_edges("verdict", route_after_scoring, branch_map)
final_builder.add_conditional_edges("adjust_and_rescore", route_after_scoring, branch_map)  # the backward arrow

# Every decision joins the summary note, then the graph ends.
final_builder.add_edge("reject", "generate_summary")
final_builder.add_edge("interview", "generate_summary")
final_builder.add_edge("strong_match", "generate_summary")
final_builder.add_edge("generate_summary", END)

final_screener_graph = final_builder.compile()
show_graph(final_screener_graph)

In [ ]:
for name, resume in RESUMES.items():
    print(f"=== {name} ===")
    final_result = final_screener_graph.invoke({"resume_text": resume, "rescore_attempts": 0, "evaluation_notes": []})
    print(f"skills={final_result['skills_score']}  experience={final_result['experience_score']}  leadership={final_result['leadership_score']}")
    print(f"final score: {final_result['score']}  |  rescore attempts: {final_result['rescore_attempts']}")
    print(f"decision: {final_result['decision']}")
    print(f"summary: {final_result['summary']}")
    for note in final_result["evaluation_notes"]:
        print("-", note)
    print()

Three LLM calls (skills, experience and leadership) ran concurrently off a single `parse_resume`, each appending its own note to `evaluation_notes` without knowing the others existed. That's only safe because `evaluation_notes` is `Annotated[list[str], operator.add]`: if it were a plain overwritten field, whichever of the three parallel writes landed last would silently erase the other two's notes. Try printing `len(final_result["evaluation_notes"])`: it's at least 5 (parse + three checks + verdict), regardless of which of the three parallel branches happened to finish first, and one more for each rescore pass.

All three candidates take different routes through the *same* graph. **Hermione** scores high and goes straight to `strong_match`. **Ron** scores low and goes to `reject`. **Harry**'s total lands in the borderline band, so he goes around Video 8's rescore loop once before landing on `interview`. Every route ends at `generate_summary`, so Priya gets a note for each candidate.

One last point before the recap: this pipeline *recommends*, and **Priya decides**. Automated screening should never be the final word on a person, which is exactly the kind of gate human-in-the-loop (Module 5) is for.

## 5. Module 1–2 recap

The Applicant Screener's arc is complete, Video 3 to here. Priya's assistant started as a single node that read a resume, and grew one complaint at a time:

| Video | Priya's problem | What it added |
|---|---|---|
| 3 | "I can't sort free-text resumes" | First node: parse resume text into structured fields |
| 4 | "Fields don't tell me whom to call" | Linear 3-node pipeline: parse, score against her rubric, summarize |
| 5 | (not part of the story) | Rebuilt with the Functional API, for comparison |
| 6 | "Why did Harry get that score?" | `evaluation_notes` as a reducer-backed audit trail |
| 7 | "They all get the same treatment" | Score-based branching: `reject` / `interview` / `strong_match`, then a summary note |
| 8 | "Borderline scores need a second look" | A bounded rescore loop for borderline scores |
| 9 | "Judge skills, experience and leadership, and make it fast" | Parallel fan-out/fan-in criteria checks, feeding routing, the rescore loop and the summary |

Every mechanic from Module 1–2 (state, nodes, edges, reducers, conditional routing, cycles, and now parallelism) is demonstrated on one running example, with the same three candidates all the way through. **Tool calling begins fresh in Video 10** with a new running example: the Meridian Robotics Financial Research Assistant.